# Module A2: Product Image Classification using MobileNetV2

This notebook demonstrates loading the Fashion-MNIST dataset, adapting the MobileNetV2 architecture for transfer learning, training the model, evaluating its performance, and saving the model weights for inference.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import matplotlib.pyplot as plt
import numpy as np
import os

print('PyTorch Version:', torch.__version__)
print('Torchvision Version:', torchvision.__version__)

## 1. Prepare Dataset
Fashion-MNIST has 10 classes of 28x28 grayscale images. We will:
1. Resize to 64x64 to allow MobileNetV2 downsampling layers to work.
2. Replicate the single grayscale channel into 3 channels.
3. Standardize using ImageNet statistics.
4. Subset the dataset for faster training demonstration.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Download datasets
train_dataset = torchvision.datasets.FashionMNIST(root='../data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform)

# Subset the datasets for fast training (e.g. 1000 train, 200 test)
train_subset = Subset(train_dataset, range(1000))
test_subset = Subset(test_dataset, range(200))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'Train subset size: {len(train_subset)}')
print(f'Test subset size: {len(test_subset)}')

## 2. Load Pretrained MobileNetV2 & Modify Classifier Layer

In [ ]:
# Load MobileNetV2 with default pre-trained ImageNet weights
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)

# Freeze backbone feature extraction layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier layer to output 10 Fashion-MNIST classes
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(classes))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print('Model classifier summary:\n', model.classifier)

## 3. Train Model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / len(train_subset)
    epoch_acc = correct / total * 100.0
    print(f'Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.2f}%')

## 4. Evaluate Model

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

accuracy = correct / total * 100.0
print(f'Test Accuracy: {accuracy:.2f}%')

## 5. Save Model Weights

In [ ]:
os.makedirs('../app/models', exist_ok=True)
model_path = '../app/models/product_classifier.pt'
torch.save(model.state_dict(), model_path)
print(f'Model saved successfully at: {model_path}')